In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import csv

In [ ]:
# run on single webpage first

URL = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/ate1-bode-melochord/"
page = requests.get(URL)

page_str = page.text

soup = BeautifulSoup(page_str, "html.parser")

# print(soup.prettify())
for p in soup.find_all("p", "bodytext"):
    print(p)

# instrument name + id are usually stored within <h> tags.
# issue: not always on the same header-level. sometimes <h1>, <h3>, ...


In [ ]:
# crawl through every items' webpage

visited_urls = set()
BASE_DOMAIN = "www.musikwissenschaft.uni-wuerzburg.de"
BASE_PATH = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scraped_text = []

def scrape(url):
    if url in visited_urls:
        return
    visited_urls.add(url)


    r = requests.get(url, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    scraped_text.append(url)
    for p in soup.find_all("p", "bodytext"):
        text = p.get_text()
        print(text)
        scraped_text.append(text)

    for tag in soup.find_all("a", href=True):

        next_url = urljoin(url, tag["href"]).split("#")[0]

        if (
            urlparse(next_url).netloc == BASE_DOMAIN
            and next_url.startswith(BASE_PATH)
            and next_url not in visited_urls
        ):
            scrape(next_url)

start_url = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scrape(start_url)

with open("uncleaned_items.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(scraped_text))


In [ ]:
with open("uncleaned_items.txt", "r") as f:
    new_text = f.read()

new_text = str(new_text)

new_text = new_text.split("Zurück zur Bestandsübersicht")

print(*new_text, sep="\n")

---

In [ ]:
# Extract indexed instruments within the collection

visited_urls = set()
BASE_DOMAIN = "www.musikwissenschaft.uni-wuerzburg.de"
BASE_PATH = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scraped_text = []

def scrape(url):
    if url in visited_urls:
        return
    visited_urls.add(url)
    print(url)

    r = requests.get(url, timeout=10)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    for p in soup.find_all("p"):
        text = p.get_text(strip=True, separator=";")
        print(text)
        scraped_text.append(text)

    for tag in soup.find_all("a", href=True):

        next_url = urljoin(url, tag["href"])

    if (
        urlparse(next_url).netloc == BASE_DOMAIN
        and next_url.startswith(BASE_PATH)
        and next_url not in visited_urls
    ):
        scrape(next_url)

start_url = "https://www.musikwissenschaft.uni-wuerzburg.de/musikinstrumente/bestand/inventarliste/"
scrape(start_url)

with open("inventory_numbers.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(scraped_text))

with open("inventory_numbers.csv", "w", encoding="utf-8", newline="") as f:
    csv.writer(f, delimiter=";").writerows([entry.lstrip(", ") for entry in line.split(";", 2)] for line in scraped_text)